In [ ]:
# Parameters
configfile = "config/config.yml"
input_data = "results/data/checkpoints/beforefilter_intermediate_spender_postmortem.pq"
targetpop_data = "results/data/checkpoints/targetpop.pq"
display_util = "workflow/scripts/display_util.py"
util = "workflow/scripts/util.py"
output_data = "results/data/intermediate_spender_postmortem.pq"
output_model = "results/data/intermediate_spender_postmortem.yml"

In [ ]:
import yaml

with open(configfile) as stream:
    config = yaml.safe_load(stream)

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib_inline

from IPython.display import Markdown
import pandera.pandas as pa
from pandera.typing import Series

matplotlib_inline.backend_inline.set_matplotlib_formats("svg")


%matplotlib inline
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", None)

sys.path.append(str(Path(util).parent))
sys.path.append(str(Path(display_util).parent))

In [ ]:
from display_util import rule_setup, display_data_doc, collist  # noqa: E402
from util import (  # noqa: E402
    drop_col_few_distinct,
    collapse_col,
    drop_duplicate_columns,
    common_translate,
    split_data,
    find_redundant_cols,
    fix_redundancies,
    SpenderID,
    replace_single_val_inplace,
)

### Target Population Filtering

The donors in the dataset were filtered to match the target population (see [](general:tpf)). Afterwards we tried again to remove empty and duplicate columns.

In [ ]:
data = pd.read_parquet(input_data)
donors = collapse_col(
    data.loc[:, ["donor_et_dso", "donor_et_id_et", "donor_et_iqtig"]],
    fun=lambda row: None if row.nunique() > 1 else row.iloc[0],
)
targetpop = pd.read_parquet(targetpop_data)
data = data[donors.isin(targetpop["donor_et_id_et"])]
display(
    Markdown(
        f"""The filter process reduced the number of donors in the data ({donors.nunique()}) and target population ({targetpop["donor_et_id_et"].nunique()})
            to {donors[donors.isin(targetpop["donor_et_id_et"])].nunique()} in the processed data.
        """
    )
)
del targetpop

In [ ]:
data = drop_col_few_distinct(data)
data = drop_duplicate_columns(data)

### Integration of Seperated Institute Data

In this file, the {term}`DSO`, {term}`IQTIG` and {term}`ET` data is already connected (see [](general:ic)) and there are no repeating donor identifiers. The following table lists the different types of rows, whhich occur in this file and which ID combination they use.

In [ ]:
assert ~donors.duplicated().any(), "Duplicated rows?"
del donors
idcols = ["donor_et_id_et", "donor_et_dso", "donor_et_iqtig"]
df = split_data(data, idcols)
summar = split_data(data, idcols, return_summar=True)

## Domain Steps

For this file the general plan for domain preprocessing was followed (see [](general:ds)).

### Row Filtering

No further filtering was necessary for this file. (see [](general:rf)).

### Unit Conversions

We applied common translations and removed unit specifiers for single unit columns (see [](general:uc)).

In [ ]:
data = common_translate(data, config["data"]["common_translations"])
cols = data.columns[data.columns.to_series().str.contains("_unit")]
assert (data[cols].nunique() != 1).sum() == 0
dropme = cols[data[cols].nunique() == 1].to_list()
display(
    Markdown(
        f"The columns {collist(dropme)} were removed as only a single unit was used."
    )
)
data.drop(columns=dropme, inplace=True)
del dropme

Some columns use 999 as an marker for invalid values. These values are replaced with the missing value indicator.

In [ ]:
replace_single_val_inplace(
    data, ["cardiac_arrest_duration_min", "hypotensive_episode_duration_min"]
)

For `liver_left_split_status` and `liver_right_split_status` the value from `liver_status` was used, if the state
was set to "Siehe Organ".

In [ ]:
liver_state = data["liver_status"]
data["liver_left_split_status"] = data["liver_left_split_status"].mask(
    lambda val: val == "Siehe Organ", liver_state
)
data["liver_right_split_status"] = data["liver_right_split_status"].mask(
    lambda val: val == "Siehe Organ", liver_state
)

### Consolidating Columns

We consolidated columns that appear for {term}`ET`, {term}`IQTIG` and {term}`DSO` (see [](general:crc))

In [ ]:
red = find_redundant_cols(data)
red.pop("donor_et")
red["donor_et_id_et"] = ["donor_et_id_et", "donor_et_dso", "donor_et_iqtig"]
fix_redundancies(data, red)

## Intermediate Dataset

In [ ]:
indcols = ["donor_et_id_et"]
data = data.sort_index(axis=1).sort_values(indcols, axis=0)
data = data.set_index(indcols)

In [ ]:
class SpenderPostmortem(SpenderID):
    abroad_visit: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Abroad Visit",
        description="Was the donor abroad?",
        isin=["no", "yes", "no evidence"],
    )
    age: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Age",
        description="What was the age of the donor?",
    )
    alcohol_abuse: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Alcohol Abuse",
        description="Was alcohol abuse diagnosed?",
        isin=["no", "yes", "unknown"],
    )
    alcohol_abuse_start_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Start Date of the alcohol abuse",
        description="When did the alcohol abuse start?",
    )
    alcohol_consumption: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Alcohol Consumption",
        description="How much alcohol did the donor consume?",
        isin=["leicht", "acceptable", "kein Konsum", "unknown", "stark"],
    )
    ambulant_treatment: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Ambulant Treatment",
        description="Was the donor in ambulant treatment?",
        isin=["no", "yes", "no evidence"],
    )
    anamnesis_by: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Anamnesis Collected By",
        description="Who filled out the anamnesis questionnaire?",
        isin=["Angehöriger", "Hausarzt"],
    )
    anamnesis_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Anamnesis Date",
        description="When was anamnesis data collected?",
    )
    anamnesis_type: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Anamnesis Type",
        description="How was anamnesis done?",
        isin=["persönlich", "telefonisch"],
    )
    autoimmune_disease: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Autoimmune disease",
        description="Was any autoimmune disease diagnosed?",
        isin=["no", "yes", "no evidence"],
    )
    birthdate: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Birthdate",
        description="When was the patient born?",
    )
    bites: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Bites",
        description="Did the donor had bites or injuries?",
        isin=["no", "yes", "no evidence"],
    )
    bleeding: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Bleedings",
        description="Did the donor had bleedings/dysmenorrhoea?",
        isin=["no", "yes", "no evidence"],
    )
    blood_donations_excluded: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Blood Donations Excluded",
        description="Was the donor exlcuded from blood donations?",
        isin=["no", "no evidence", "yes"],
    )
    blood_transfusions: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Blood Transfusions",
        description="Did the donor received any blood transfusions?",
        isin=["no", "yes", "no evidence"],
    )
    bloodgroup: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Bloodgroup",
        description="What was the donors bloodgroup?",
        isin=["A", "B", "AB", "0"],
    )
    brain_death_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Brain Death Date",
        description="When was the donors brain death?",
    )
    cardiac_arrest: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Cardiac Arrest",
        description="Had the donor a cardiac arrest?",
        isin=["no", "yes", "no evidence"],
    )
    cardiac_arrest_duration_min: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Cardiac Arrest Duration",
        description="How long did the donors cardiac arrest last in min?",
    )
    collection_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Collection Date",
        description="When was the DSO requested for a organ donation?",
    )
    colon_malignancy: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Colon Cancer",
        description="Had the donor colon cancer?",
        isin=["no", "yes", "no evidence"],
    )
    colon_state: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Colon Donor",
        description="Could the colon be donated?",
        isin=["Abbruch vor ET-Meldung", "Nicht vermittelbar", "Organ zurückgezogen"],
    )
    coordination_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Coordination Date",
        description="When did the DSO coordinate the organ donation?",
    )
    coronary_artery_disease: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Coronary Artery Disease",
        description="Had the donor a coronary artery disease?",
        isin=["no", "yes", "no evidence"],
    )
    criminal_history: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Criminal History",
        description="Had the donor a criminal history?",
        isin=["no", "yes", "no evidence"],
    )
    death_reason: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Death Reason",
        description="Why did the donor die?",
    )
    diabetes_dso: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Diabetes",
        description="Did the donor had diabetes?",
    )
    diabetes_start_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Diabetes Start Date",
        description="When did the donors diabetes start?",
    )
    diabetes_type: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Diabetes Type",
        description="What type of diabetes did the donor had?",
    )
    diarrhea: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Diarrhea",
        description="Had the donor diarrhea?",
        isin=["no", "yes", "no evidence"],
    )
    disease_pre_operations: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Pre-existing Illnesses or Pre-operations",
        description="Had the donor pre_existing illnesses or pre-operations?",
        isin=["no", "yes", "no evidence"],
    )
    doctor_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Doctor Date",
        description="When was the donor treated?",
    )
    donor_abroad: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Donor Abroad",
        description="Was the donor from a foreign country?",
        isin=["no", "yes", "no evidence"],
    )
    donor_status: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Donor status",
        description="Were a donors organs reserved for a recipient?",
        isin=["vollständig vermittelt"],
    )
    donor_type: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Donor Type",
        description="What type was this donor?",
        isin=["Cadaver"],
    )
    donor_type_iqtig: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Donor Type According to IQTIG",
        description="What type was this donor according to the IQTIG?",
        isin=["hirntot"],
    )
    drug_abuse: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Drug Abuse",
        description="Did the donor abuse drugs?",
        isin=["no", "yes", "unknown"],
    )
    drug_use: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Drug User",
        description="How did the donor take drugs?",
        isin=[
            "kein Konsum",
            "unknown",
            "oral",
            "intravenös in den letzten 3 Monaten",
            "intravenös",
            "ehemals intravenös",
        ],
    )
    frequent_sex_partner_change: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Frequent Change of Sex Parnters",
        description="Did the donor change sex parnters frequently?",
        isin=["no", "yes", "no evidence"],
    )
    gynacologic_malignancy: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Gynacological Cancer",
        description="Had the donor had gynacological cancer?",
        isin=["no", "yes", "no evidence"],
    )
    headaches: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Headaches",
        description="Had the donor headaches?",
        isin=["no", "yes", "no evidence"],
    )
    heart_disease: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Heart Disease",
        description="Had the donor a heart disease?",
        isin=["no", "yes", "no evidence"],
    )
    heart_state: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Heart Donor",
        description="Could the heart be donated?",
        isin=[
            "Abbruch vor ET-Meldung",
            "Nicht vermittelbar",
            "Nicht transplantiert",
            "Organ zurückgezogen",
        ],
    )
    height_cm: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Height",
        description="What was the donors height in cm?",
    )
    hematologic_disease: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Hematologic Disease",
        description="Had the donor a hematologic disease?",
        isin=["no", "yes", "no evidence"],
    )
    hepatitis: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Hepatitis",
        description="Had the donor hepatitis?",
        isin=["no", "yes", "no evidence"],
    )
    high_risk: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="High Risk",
        description="Was the donor considered high risk?",
        isin=["no", "yes", "no evidence"],
    )
    hiv_high_risk_visit: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="HIV High Risk Area Visit",
        description="Had the donor a long term visit in an high risk HIV area?",
        isin=["no", "yes", "no evidence"],
    )
    hospitalization_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Hospitalization Date",
        description="When was the donor hospitialized?",
    )
    hypertension: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Hypertension",
        description="Had the donor hypertension?",
        isin=["no", "yes", "unknown", "no evidence"],
    )
    hypertension_start_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Begin of Hypertension",
        description="When did the donors hypertension start?",
    )
    hypertension_treated: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Hypertension Treated",
        description="Was the donors hypertension treated?",
        isin=["no", "yes", "no evidence"],
    )
    hypotensive_episode: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Hypotensive Episode",
        description="Had the donor a hypotensive episode?",
        isin=["no", "yes", "no evidence"],
    )
    hypotensive_episode_duration_min: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Hypotensive Episode Duration",
        description="How long did the hypotensive episode last in min?",
    )
    infectious_diseases: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Infectious Diseases",
        description="Had the donor infectious diseases?",
        isin=["no", "yes", "no evidence"],
    )
    injection: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Injection",
        description="Had the donor an injection without medical indication?",
        isin=["no", "yes", "no evidence"],
    )
    intensive_care_start_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Begin of Intensive Care",
        description="When did the intensive care start?",
    )
    interviewer: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Interviewer",
        description="Who did the anamnesis?",
        isin=["Arzt", "Beide", "Koordinator"],
    )
    kidney_left_status: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Left Kidney Donor",
        description="Could the left kidney be donated?",
        isin=[
            "Abbruch vor ET-Meldung",
            "Nicht transplantiert",
            "Nicht vermittelbar",
            "Organ zurückgezogen",
        ],
    )
    kidney_right_status: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Right Kidney Donor",
        description="Could the right kidney be donated?",
        isin=[
            "Abbruch vor ET-Meldung",
            "Nicht transplantiert",
            "Nicht vermittelbar",
            "Organ zurückgezogen",
        ],
    )
    last_reanimation_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Last Reanimation Date",
        description="When was the last reanimation performed?",
    )
    law_colon_allowed: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Legal Colon Donor",
        description="Was the donor legally allowed to donate their colon?",
        isin=["yes", "no"],
    )
    law_decision_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Legal Decision Date",
        description="When were the legal decisions for this donor made?",
    )
    law_heart_allowed: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Legal Heart Donor",
        description="Was the donor legally allowed to donate their heart?",
        isin=["yes", "no"],
    )
    law_kidney_left_allowed: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Legal Left Kidney Donor",
        description="Was the donor legally allowed to donate their left kidney?",
        isin=["yes", "no"],
    )
    law_kidney_right_allowed: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Legal Right Kidney Donor",
        description="Was the donor legally allowed to donate their right kidney?",
        isin=["yes", "no"],
    )
    law_liver_allowed: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Legal Liver Donor",
        description="Was the donor legally allowed to donate their liver?",
        isin=["yes", "no"],
    )
    law_kidney_left_allowed: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Legal Left Lung Donor",
        description="Was the donor legally allowed to donate their left lung?",
        isin=["yes", "no"],
    )
    law_liver_allowed: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Legal Pancreas Donor",
        description="Was the donor legally allowed to donate their pancreas?",
        isin=["yes", "no"],
    )
    liver_left_split_status: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Liver Left Split Donor",
        description="Could the left liver split be donated?",
        isin=[
            "Abbruch vor ET-Meldung",
            "Nicht transplantiert",
            "Nicht vermittelbar",
            "Organ zurückgezogen",
        ],
    )
    liver_right_split_status: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Liver Right Split Donor",
        description="Could the right liver split be donated?",
        isin=[
            "Abbruch vor ET-Meldung",
            "Nicht transplantiert",
            "Nicht vermittelbar",
            "Organ zurückgezogen",
        ],
    )
    liver_status: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Liver Donor",
        description="Could the liver be donated?",
        isin=[
            "Abbruch vor ET-Meldung",
            "Nicht transplantiert",
            "Nicht vermittelbar",
            "Organ zurückgezogen",
            "Split",
        ],
    )
    lung_left_status: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Left Lung Donor",
        description="Could the left lung be donated?",
        isin=[
            "Abbruch vor ET-Meldung",
            "Nicht transplantiert",
            "Nicht vermittelbar",
            "Organ zurückgezogen",
        ],
    )
    lung_right_status: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Right Lung Donor",
        description="Could the right lung be donated?",
        isin=[
            "Abbruch vor ET-Meldung",
            "Nicht transplantiert",
            "Nicht vermittelbar",
            "Organ zurückgezogen",
        ],
    )
    lymph_nodes_swollen: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Swollen Lymph Nodes",
        description="Were the donors lymph nodes swollen?",
        isin=["no", "yes", "no evidence"],
    )
    male_homosexual_activity: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Male Homosexual Activity",
        description="Had this male donor intercourse with a male partner?",
        isin=["no", "yes", "no evidence"],
    )
    malignancy: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Cancer",
        description="Had the donor cancer?",
    )
    malignancy_details: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Cancer Details",
        description="How was the cancer described?",
    )
    mamma_malginity: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Breast Cancer",
        description="Had the donor breast cancer?",
        isin=["no", "yes", "no evidence"],
    )
    medication: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Medication",
        description="Did the donor take medications regularly?",
        isin=["no", "yes", "no evidence"],
    )
    melanom_malignancy: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Melanoma",
        description="Had the donor a melanoma?",
        isin=["no", "yes", "no evidence"],
    )
    mother_hiv_positive: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Mother HIV Positive",
        description="Had the donors mother HIV?",
        isin=["no", "yes", "no evidence"],
    )
    mrsa: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="MRSA",
        description="Had the donor a MRSA infection?",
        isin=["no", "yes", "no evidence"],
    )
    nationality: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Nationality",
        description="What was the nationality of the donor?",
    )
    neurologic_disease: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Neurologic Disease",
        description="Had the donor a neurologic disease?",
        isin=["no", "yes", "no evidence"],
    )
    night_sweats: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Night Sweats",
        description="Had the donor night sweats?",
        isin=["no", "yes", "no evidence"],
    )
    organ_removal_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Organ Removal Date",
        description="When were the organs removed?",
    )
    other_diseases: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Other Diseases",
        description="Had the donor other diseases?",
        isin=["no", "yes", "no evidence"],
    )
    other_diseases_reason_unknown: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Other Diseases with Unknown Reason",
        description="What other diseases with unknown reasons did the donor have?",
        isin=["no", "yes", "no evidence"],
    )
    other_infectious_diseases: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Other Infectious Diseases",
        description="Had the donor other infectious diseases?",
        isin=["no", "yes", "no evidence"],
    )

    other_malignancy: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Other Cancer",
        description="Had the donor other cancers?",
        isin=["no", "yes", "no evidence"],
    )
    pain_medication: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Pain Medication",
        description="Had the donor received pain medication?",
        isin=["no", "yes", "no evidence"],
    )
    pancreas_state: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Pancreas Donor",
        description="Could the pancreas be donated?",
        isin=[
            "Abbruch vor ET-Meldung",
            "Nicht vermittelbar",
            "Organ zurückgezogen",
            "Inselzellen",
            "Nicht transplantiert",
        ],
    )
    pre_operation: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Pre-operation",
        description="Had the donor a pre-operation?",
        isin=["no", "yes", "no evidence"],
    )
    prosecutor_colon_allowed: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Approved Legal Colon Donor",
        description="Was the donor legally allowed by the prosecutor to donate their colon?",
        isin=["yes", "no"],
    )
    prosecutor_decision_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Prosecutor Decision Date",
        description="When did the prosecutor make their decision?",
    )
    prosecutor_heart_allowed: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Approved Legal Heart Donor",
        description="Was the donor legally allowed by the prosecutor to donate their heart?",
        isin=["yes", "no"],
    )
    prosecutor_kidney_left_allowed: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Approved Legal Left Kidney Donor",
        description="Was the donor legally allowed by the prosecutor to donate their left kidney?",
        isin=["yes", "no"],
    )
    prosecutor_kidney_right_allowed: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Approved Legal Right Kidney Donor",
        description="Was the donor legally allowed by the prosecutor to donate their right kidney?",
        isin=["yes", "no"],
    )
    prosecutor_liver_allowed: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Approved Legal Liver Donor",
        description="Was the donor legally allowed by the prosecutor to donate their liver?",
        isin=["yes", "no"],
    )
    prosecutor_lung_left_allowed: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Approved Legal Left Lung Donor",
        description="Was the donor legally allowed by the prosecutor to donate their left lung?",
        isin=["yes", "no"],
    )
    prosecutor_liver_allowed: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Approved Legal Liver Donor",
        description="Was the donor legally allowed by the prosecutor to donate their liver?",
        isin=["yes", "no"],
    )
    prostate_malignancy: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Prostate cancer",
        description="Indicator if donor had prostate cancer",
        isin=["no", "yes", "no evidence"],
    )
    prostitution: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Prostitution",
        description="Did the donor prostitute?",
        isin=["no", "yes", "no evidence"],
    )
    reanimation_cpimnt: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Number of Reanimations",
        description="How many reanimations did the donor have?",
    )
    reanimation_duration_min: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Reanimation Duration",
        description="How long was the last reanimation in min?",
    )
    received_human_clotting: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Received Human Clotting",
        description="Did the donor receive human clotting agents?",
        isin=["no", "yes", "no evidence"],
    )
    recent_abnormalities: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Recent Abnormalities",
        description="Had the donor recent abnormalities?",
        isin=["no", "yes", "no evidence"],
    )
    registry_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Registry Date",
        description="When was the donor registered at the ET?",
    )
    relative_blood_donor: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Relative Blood Donor",
        description="Was a relative a blood donor?",
        isin=["no", "yes", "no evidence"],
    )
    rhesus_factor: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Rhesus Factor",
        description="What was the rhesus factor of donor?",
        isin=["positive", "negative"],
    )
    sex: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Sex",
        description="What was the donors biological sex?",
        isin=["male", "female"],
    )
    sex_partner_stds: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="STDs of Sex Parnters",
        description="Had the donor sex partners with STDs?",
        isin=["no", "yes", "no evidence"],
    )
    smoker: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Smoker",
        description="Was the donor a smoker?",
        isin=["no", "yes", "no evidence", "unknown"],
    )
    smoker_pack_years: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Smoker Pack Years",
        description="How much did the donor smoke in pack years?",
    )
    smoking_amount: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Amount of Smoking",
        description="How much did the donor smoke?",
        isin=["kein Konsum", "stark", "leicht", "unknown", "acceptable"],
    )
    stationary_treatment: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Stationary Treatment",
        description="Was the treatment of the donor stationary?",
        isin=["no", "yes", "no evidence"],
    )
    stds: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="STDs",
        description="Had the donors any STDs?",
        isin=["no", "yes", "no evidence"],
    )
    tattoo_piercings: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Tattoos, Piercings or Punctures",
        description="Did the donors have any tattoos, piercings or other punctures?",
        isin=["no", "yes", "no evidence"],
    )
    tattoo_piercings_sterile: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Sterile Puncture",
        description="Had the donor punctures with sterile needles?",
        isin=["no", "yes", "no evidence"],
    )
    treated_by_doctor: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Treated by Doctor",
        description="Was the donor treated by a doctor?",
        isin=["no", "yes", "no evidence"],
    )
    treated_diabetes: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Treated Diabetes",
        description="Was the donors diabetes treated?",
        isin=["no", "yes", "no evidence"],
    )
    tubercolosis: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Tubercolosis",
        description="Had the donor tubercolosis?",
        isin=["no", "yes", "no evidence"],
    )
    type_cadaver: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Cadaver Type",
        description="What kind of cadaver was observed?",
        isin=["Heart Beating"],
    )
    urine_catheter_start_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Urine Catheter Start Date",
        description="When did the donor start using a urine catheter?",
    )
    vaccinated: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Vaccinated",
        description="Was the donor vaccinated?",
        isin=["no", "yes", "no evidence"],
    )
    ventilation_duration_hours: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Ventilation Duration",
        description="How long did the donor receive ventilation for in hours?",
    )
    ventilation_start_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Begin of Ventilation",
        description="When was the donors ventilation started?",
    )
    weight_kg: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Weight",
        description="How much did the donor weigh in kg?",
    )
    weight_reduction: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Weight Reduction",
        description="Did the donor loose weight?",
        isin=["no", "yes", "no evidence"],
    )

    class Config:
        title = "Deceased Donor Dataset"
        description = "Each row represents base data on the deceased donors. The data is based on the 'element_spender_postmortem.csv' file. It contains data from the ET, DSO and IQTIG."
        multiindex_strict = True
        multiindex_coerce = True

In [ ]:
display_data_doc(SpenderPostmortem, data)

In [ ]:
SpenderPostmortem.to_schema().validate(data).to_parquet(output_data)
with open(output_model, "wt") as fh:
    SpenderPostmortem.to_yaml(stream=fh)

## Technical Information

In [ ]:
rule_setup(
    {
        "input_data": input_data,
        "targetpop_data": targetpop_data,
        "output_data": output_data,
        "output_model": output_model,
        "util": util,
        "display_util": display_util,
    }
)